# Fast Candidate-Generator Oracle Lab

        This misc notebook is a fast lab for candidate-generator work. It is intentionally upstream of model training: the question is whether a generator proposes a plausible RPF window before any XGB scorer tries to choose one.

        Default mode is `focus`, which runs only on selected Beta failure cases for fast iteration. Set `RUN_MODE = "full"` or environment variable `CANDIDATE_LAB_RUN_MODE=full` to regenerate complete Alpha/Beta oracle metrics.

## 1. Controls And Setup

        Focus mode is the default because candidate-generator experiments need quick turnaround. Full mode remains available for complete summaries after an idea looks promising.

In [ ]:
from __future__ import annotations

import json
import os
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots

RUN_MODE = os.environ.get("CANDIDATE_LAB_RUN_MODE", "focus").strip().lower()
if RUN_MODE not in {"focus", "full"}:
    raise ValueError("RUN_MODE must be 'focus' or 'full'.")

FOCUS_SITES = ["beta_B", "beta_D", "beta_E"]
FOCUS_FAILURE_GROUPS = ["no_candidate", "boundary_mismatch", "poor_overlap"]
EXAMPLES_PER_SITE_GROUP = 5
WRITE_HTML = True
CLEAN_HTML_OUTPUTS = True
WRITE_FULL_DAY_SUMMARY = RUN_MODE == "full"
REQUIRED_EXAMPLE_DAYS = [("beta_B", "2023-10-13")]

SEARCH_START_HOUR = 6
SEARCH_END_HOUR = 18
MIN_DURATION_MINUTES = 30
MAX_DURATION_MINUTES = 8 * 60
STRICT_BOUNDARY_TOLERANCE_MINUTES = 30
IOU_HIT_THRESHOLD = 0.50

PALETTE = {
    "orange": "#eb932c",
    "dark_blue": "#22303d",
    "grey": "#2F4D67",
    "light_grey": "#5C7D99",
    "light_white": "#ebe3e3",
}
plt.rcParams.update({
    "font.family": "Arial",
    "axes.edgecolor": PALETTE["dark_blue"],
    "axes.labelcolor": PALETTE["dark_blue"],
    "axes.titlecolor": PALETTE["dark_blue"],
    "xtick.color": PALETTE["dark_blue"],
    "ytick.color": PALETTE["dark_blue"],
})

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "publication" / "2_journal_article" / "dataset" / "final").exists():
            return candidate
    raise FileNotFoundError("Could not find PyNRPF repo root.")

REPO_ROOT = find_repo_root()
ARTICLE_ROOT = REPO_ROOT / "publication" / "2_journal_article"
MISC_DIR = ARTICLE_ROOT / "notebooks" / "99_Misc"
M9_OUTPUT = MISC_DIR / "outputs" / "03_m9_hybrid_development"
OUTPUT_ROOT = MISC_DIR / "outputs" / "06_candidate_generator_oracle_analysis"
CSV_DIR = OUTPUT_ROOT / "csv"
FIGURE_DIR = OUTPUT_ROOT / "figures"
HTML_DIR = OUTPUT_ROOT / "html_examples"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [CSV_DIR, FIGURE_DIR, HTML_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
if CLEAN_HTML_OUTPUTS:
    for old_html in HTML_DIR.glob("*.html"):
        old_html.unlink()

OUTPUT_PREFIX = "focus_" if RUN_MODE == "focus" else ""
t0 = time.perf_counter()
print("RUN_MODE:", RUN_MODE)
print("Output root:", OUTPUT_ROOT)

## 2. Load Data And Existing m7 Candidates

        This cell loads final Alpha/Beta datasets and the existing m7-style candidates. In focus mode, only Beta focus sites are used after the current-m7 failure set is identified.

In [ ]:
def naive_timestamp(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce").dt.tz_convert(None)

def load_final_dataset(name: str) -> pd.DataFrame:
    path = ARTICLE_ROOT / "dataset" / "final" / f"dataset_{name}.parquet"
    df = pd.read_parquet(path)
    expected = ["substation_id", "date", "timestamp", "net_load_MW", "solar_MW", "label_interval", "label_day"]
    missing = [col for col in expected if col not in df.columns]
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")
    df = df[expected].copy()
    df["timestamp"] = naive_timestamp(df["timestamp"])
    df["date"] = df["date"].astype(str)
    df["label_interval"] = df["label_interval"].astype(bool)
    df["label_day"] = df["label_day"].astype(bool)
    return df.sort_values(["substation_id", "timestamp"]).reset_index(drop=True)

def load_m7_candidates(name: str) -> pd.DataFrame:
    filename = "01_alpha_m7_candidates.csv" if name == "alpha" else "05_beta_m7_candidates.csv"
    path = M9_OUTPUT / "intermediate" / filename
    cand = pd.read_csv(path)
    cand["date"] = cand["date"].astype(str)
    for col in ["pred_start", "pred_end", "peak_time", "solar_peak_time"]:
        if col in cand.columns:
            cand[col] = naive_timestamp(cand[col])
    cand = cand.loc[cand["candidate_status"].eq("candidate") & cand["pred_start"].notna() & cand["pred_end"].notna()].copy()
    cand["candidate_source"] = "current_m7"
    return cand[["substation_id", "date", "candidate_source", "candidate_id", "pred_start", "pred_end"]].reset_index(drop=True)

alpha = load_final_dataset("alpha") if RUN_MODE == "full" else None
beta = load_final_dataset("beta")
alpha_m7 = load_m7_candidates("alpha") if RUN_MODE == "full" else None
beta_m7 = load_m7_candidates("beta")

print("Beta rows/sites/days:", len(beta), beta["substation_id"].nunique(), beta[["substation_id", "date"]].drop_duplicates().shape[0])
if alpha is not None:
    print("Alpha rows/sites/days:", len(alpha), alpha["substation_id"].nunique(), alpha[["substation_id", "date"]].drop_duplicates().shape[0])

## 3. Cached Site-Day Representation

        The `day_cache` is the main speed improvement. It stores arrays and true-window metadata once per site-day, so every generator can reuse the same precomputed structure instead of repeatedly grouping and re-reading raw data.

In [ ]:
@dataclass
class DayRecord:
    site: str
    date: str
    frame: pd.DataFrame
    ts: pd.Series
    net: np.ndarray
    solar: np.ndarray
    pseudo: np.ndarray
    ds: np.ndarray
    dn: np.ndarray
    daytime: np.ndarray
    label_day: bool
    true_start: pd.Timestamp | pd.NaT
    true_end: pd.Timestamp | pd.NaT
    true_interval_count: int
    solar_peak: float

def build_day_cache(raw: pd.DataFrame, keys: set[tuple[str, str]] | None = None, sites: list[str] | None = None) -> dict[tuple[str, str], DayRecord]:
    df = raw
    if sites is not None:
        df = df.loc[df["substation_id"].isin(sites)]
    if keys is not None:
        key_index = pd.MultiIndex.from_frame(df[["substation_id", "date"]])
        df = df.loc[key_index.isin(pd.MultiIndex.from_tuples(sorted(keys), names=["substation_id", "date"]))]

    cache = {}
    for (site, date), day in df.groupby(["substation_id", "date"], sort=False):
        day = day.sort_values("timestamp").reset_index(drop=True)
        ts = day["timestamp"]
        net = day["net_load_MW"].to_numpy(dtype=float)
        solar = day["solar_MW"].to_numpy(dtype=float)
        pseudo = solar - net
        labelled = day.loc[day["label_interval"]]
        label_day = bool(len(labelled))
        true_start = labelled["timestamp"].iloc[0] if label_day else pd.NaT
        true_end = labelled["timestamp"].iloc[-1] if label_day else pd.NaT
        solar_peak = float(np.nanmax(solar)) if np.isfinite(solar).any() else np.nan
        cache[(str(site), str(date))] = DayRecord(
            site=str(site),
            date=str(date),
            frame=day,
            ts=ts,
            net=net,
            solar=solar,
            pseudo=pseudo,
            ds=np.diff(solar, prepend=np.nan),
            dn=np.diff(net, prepend=np.nan),
            daytime=ts.dt.hour.between(SEARCH_START_HOUR, SEARCH_END_HOUR, inclusive="both").to_numpy(),
            label_day=label_day,
            true_start=true_start,
            true_end=true_end,
            true_interval_count=int(len(labelled)),
            solar_peak=solar_peak,
        )
    return cache

def windows_from_cache(cache: dict[tuple[str, str], DayRecord]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "substation_id": rec.site,
            "date": rec.date,
            "label_day": rec.label_day,
            "true_start": rec.true_start,
            "true_end": rec.true_end,
            "true_interval_count": rec.true_interval_count,
        }
        for rec in cache.values()
    ])

beta_focus_site_cache = build_day_cache(beta, sites=FOCUS_SITES)
beta_focus_windows = windows_from_cache(beta_focus_site_cache)
print("Cached Beta focus site-days:", len(beta_focus_site_cache))

## 4. Fast Oracle Evaluation

        Candidate evaluation now pre-groups candidates by `(substation_id, date)` once. This removes the old repeated full-table filtering inside every site-day loop.

In [ ]:
def interval_iou(a0: pd.Timestamp, a1: pd.Timestamp, b0: pd.Timestamp, b1: pd.Timestamp) -> float:
    if pd.isna(a0) or pd.isna(a1) or pd.isna(b0) or pd.isna(b1):
        return 0.0
    a1_ex = a1 + pd.Timedelta(minutes=15)
    b1_ex = b1 + pd.Timedelta(minutes=15)
    overlap = max(pd.Timedelta(0), min(a1_ex, b1_ex) - max(a0, b0)).total_seconds() / 60
    union = (max(a1_ex, b1_ex) - min(a0, b0)).total_seconds() / 60
    return float(overlap / union) if union > 0 else 0.0

def candidate_group_lookup(candidates: pd.DataFrame) -> dict[tuple[str, str], pd.DataFrame]:
    if candidates is None or candidates.empty:
        return {}
    cand = candidates.copy()
    cand["date"] = cand["date"].astype(str)
    for col in ["pred_start", "pred_end"]:
        cand[col] = pd.to_datetime(cand[col], errors="coerce")
    return {key: grp.copy() for key, grp in cand.groupby(["substation_id", "date"], sort=False)}

def evaluate_candidates(cache: dict[tuple[str, str], DayRecord], candidates: pd.DataFrame, dataset: str, variant_id: str):
    grouped = candidate_group_lookup(candidates)
    day_rows = []
    for key, rec in cache.items():
        cc = grouped.get(key)
        candidate_count = 0 if cc is None else len(cc)
        strict_hit = False
        iou_hit = False
        best_iou = 0.0
        best_start_error = np.nan
        best_end_error = np.nan
        best_candidate_id = np.nan
        if rec.label_day and cc is not None and not cc.empty:
            best_rows = []
            for _, c in cc.iterrows():
                start_error = abs((c["pred_start"] - rec.true_start).total_seconds()) / 60
                end_error = abs((c["pred_end"] - rec.true_end).total_seconds()) / 60
                iou = interval_iou(c["pred_start"], c["pred_end"], rec.true_start, rec.true_end)
                best_rows.append((iou, -start_error - end_error, c.get("candidate_id", np.nan), start_error, end_error))
                strict_hit = strict_hit or (start_error <= STRICT_BOUNDARY_TOLERANCE_MINUTES and end_error <= STRICT_BOUNDARY_TOLERANCE_MINUTES)
                iou_hit = iou_hit or (iou >= IOU_HIT_THRESHOLD)
            best = sorted(best_rows, reverse=True)[0]
            best_iou = float(best[0])
            best_candidate_id = best[2]
            best_start_error = float(best[3])
            best_end_error = float(best[4])
        day_rows.append({
            "dataset": dataset,
            "variant_id": variant_id,
            "substation_id": rec.site,
            "date": rec.date,
            "label_day": rec.label_day,
            "candidate_count": int(candidate_count),
            "candidate_day": bool(candidate_count > 0),
            "strict_hit": bool(strict_hit),
            "iou50_hit": bool(iou_hit),
            "best_iou": best_iou,
            "best_start_error_minutes": best_start_error,
            "best_end_error_minutes": best_end_error,
            "best_candidate_id": best_candidate_id,
        })
    day = pd.DataFrame(day_rows)

    def summarise(group: pd.DataFrame, site: str | None = None) -> dict:
        rpf = group.loc[group["label_day"]]
        non = group.loc[~group["label_day"]]
        return {
            "dataset": dataset,
            "variant_id": variant_id,
            "substation_id": site or "ALL",
            "site_days": int(len(group)),
            "rpf_days": int(len(rpf)),
            "strict_hits": int(rpf["strict_hit"].sum()) if len(rpf) else 0,
            "iou50_hits": int(rpf["iou50_hit"].sum()) if len(rpf) else 0,
            "strict_recall": float(rpf["strict_hit"].mean()) if len(rpf) else np.nan,
            "iou50_recall": float(rpf["iou50_hit"].mean()) if len(rpf) else np.nan,
            "mean_best_iou_rpf": float(rpf["best_iou"].mean()) if len(rpf) else np.nan,
            "median_best_iou_rpf": float(rpf["best_iou"].median()) if len(rpf) else np.nan,
            "total_candidates": int(group["candidate_count"].sum()),
            "candidate_days": int(group["candidate_day"].sum()),
            "candidate_day_rate": float(group["candidate_day"].mean()) if len(group) else np.nan,
            "non_rpf_candidate_days": int(non["candidate_day"].sum()) if len(non) else 0,
            "non_rpf_candidate_day_rate": float(non["candidate_day"].mean()) if len(non) else np.nan,
            "mean_candidates_per_day": float(group["candidate_count"].mean()) if len(group) else np.nan,
            "mean_candidates_per_candidate_day": float(group.loc[group["candidate_day"], "candidate_count"].mean()) if group["candidate_day"].any() else 0.0,
        }

    overall = pd.DataFrame([summarise(day)])
    by_site = pd.DataFrame([summarise(g, site) for site, g in day.groupby("substation_id", sort=True)])
    return overall, by_site, day

def filter_candidates_to_cache(candidates: pd.DataFrame, cache: dict[tuple[str, str], DayRecord]) -> pd.DataFrame:
    if candidates is None or candidates.empty:
        return pd.DataFrame(columns=["substation_id", "date", "candidate_source", "candidate_id", "pred_start", "pred_end"])
    keys = set(cache)
    temp = candidates.copy()
    temp["date"] = temp["date"].astype(str)
    mask = list(zip(temp["substation_id"], temp["date"]))
    return temp.loc[[key in keys for key in mask]].copy().reset_index(drop=True)

## 5. Candidate Generators

        These functions consume cached site-days. The plateau-aware generator is the main experiment: it treats a broad/flat local maximum as a valid RPF bump peak and then reuses m7-style left/right minima to create boundaries.

In [ ]:
def candidate_row(rec: DayRecord, source: str, candidate_id: int, start: pd.Timestamp, end: pd.Timestamp, extra: dict | None = None) -> dict | None:
    if pd.isna(start) or pd.isna(end) or end < start:
        return None
    duration = (end - start).total_seconds() / 60 + 15
    if duration < MIN_DURATION_MINUTES or duration > MAX_DURATION_MINUTES:
        return None
    if start.hour < SEARCH_START_HOUR or end.hour > SEARCH_END_HOUR:
        return None
    row = {
        "substation_id": rec.site,
        "date": rec.date,
        "candidate_source": source,
        "candidate_id": candidate_id,
        "pred_start": start,
        "pred_end": end,
        "duration_minutes": duration,
    }
    if extra:
        row.update(extra)
    return row

def runs_from_mask(mask: np.ndarray) -> list[np.ndarray]:
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return []
    splits = np.where(np.diff(idx) > 1)[0] + 1
    return [part for part in np.split(idx, splits) if len(part)]

def fill_one_step_gaps(mask: np.ndarray) -> np.ndarray:
    out = mask.copy()
    for i in range(1, len(mask) - 1):
        if not out[i] and out[i - 1] and out[i + 1]:
            out[i] = True
    return out

def nearest_window_from_center(rec: DayRecord, center: pd.Timestamp, duration_minutes: int):
    half = pd.Timedelta(minutes=duration_minutes / 2)
    start_target = center - half
    end_target = center + half - pd.Timedelta(minutes=15)
    valid = rec.ts.loc[rec.ts.dt.hour.between(SEARCH_START_HOUR, SEARCH_END_HOUR, inclusive="both")]
    if valid.empty:
        return None
    start = valid.iloc[np.argmin(np.abs((valid - start_target).dt.total_seconds().to_numpy()))]
    end = valid.iloc[np.argmin(np.abs((valid - end_target).dt.total_seconds().to_numpy()))]
    return start, end

def generate_solar_peak_windows(cache: dict[tuple[str, str], DayRecord], min_peak_solar: float = 2.0) -> pd.DataFrame:
    rows = []
    for rec in cache.values():
        if not np.isfinite(rec.solar_peak) or rec.solar_peak < min_peak_solar:
            continue
        valid = rec.daytime & np.isfinite(rec.solar)
        if not valid.any():
            continue
        peak_idx = int(np.flatnonzero(valid)[np.nanargmax(rec.solar[valid])])
        peak_time = rec.ts.iloc[peak_idx]
        cid = 0
        for duration in [60, 120, 180, 240]:
            window = nearest_window_from_center(rec, peak_time, duration)
            if window is None:
                continue
            row = candidate_row(rec, "solar_peak_windows", cid, window[0], window[1])
            if row:
                rows.append(row)
                cid += 1
    return pd.DataFrame(rows)

def generate_solar_high_segments(cache: dict[tuple[str, str], DayRecord], min_peak_solar: float = 2.0, frac_peak: float = 0.45) -> pd.DataFrame:
    rows = []
    for rec in cache.values():
        if not np.isfinite(rec.solar_peak) or rec.solar_peak < min_peak_solar:
            continue
        mask = rec.daytime & (rec.solar >= max(min_peak_solar, frac_peak * rec.solar_peak))
        cid = 0
        for run in runs_from_mask(mask):
            start_idx = max(int(run[0]) - 1, 0)
            end_idx = min(int(run[-1]) + 1, len(rec.ts) - 1)
            row = candidate_row(rec, "solar_high_segments", cid, rec.ts.iloc[start_idx], rec.ts.iloc[end_idx])
            if row:
                rows.append(row)
                cid += 1
    return pd.DataFrame(rows)

def generate_comovement_segments(cache: dict[tuple[str, str], DayRecord], min_peak_solar: float = 2.0) -> pd.DataFrame:
    rows = []
    for rec in cache.values():
        if not np.isfinite(rec.solar_peak) or rec.solar_peak < min_peak_solar:
            continue
        same = np.isfinite(rec.ds) & np.isfinite(rec.dn) & (np.abs(rec.ds) > 0.02) & (np.abs(rec.dn) > 0.02) & ((rec.ds * rec.dn) > 0)
        same_roll = pd.Series(same.astype(float)).rolling(3, center=True, min_periods=1).mean().to_numpy() >= 0.5
        solar_ok = rec.solar >= max(1.0, 0.20 * rec.solar_peak)
        mask = fill_one_step_gaps(rec.daytime & solar_ok & same_roll)
        cid = 0
        for run in runs_from_mask(mask):
            row = candidate_row(rec, "comovement_segments", cid, rec.ts.iloc[int(run[0])], rec.ts.iloc[int(run[-1])])
            if row:
                rows.append(row)
                cid += 1
    return pd.DataFrame(rows)

def generate_pseudoload_stable_segments(cache: dict[tuple[str, str], DayRecord], min_peak_solar: float = 2.0) -> pd.DataFrame:
    rows = []
    for rec in cache.values():
        if not np.isfinite(rec.solar_peak) or rec.solar_peak < min_peak_solar:
            continue
        slope = np.abs(np.diff(rec.pseudo, prepend=np.nan))
        rolling_slope = pd.Series(slope).rolling(5, center=True, min_periods=2).mean().to_numpy()
        solar_ok = rec.solar >= max(1.0, 0.25 * rec.solar_peak)
        ref = rolling_slope[rec.daytime & solar_ok & np.isfinite(rolling_slope)]
        if len(ref) < 4:
            continue
        threshold = np.nanpercentile(ref, 35)
        mask = fill_one_step_gaps(rec.daytime & solar_ok & (rolling_slope <= threshold))
        cid = 0
        for run in runs_from_mask(mask):
            row = candidate_row(rec, "pseudoload_stable_segments", cid, rec.ts.iloc[int(run[0])], rec.ts.iloc[int(run[-1])])
            if row:
                rows.append(row)
                cid += 1
    return pd.DataFrame(rows)

def generate_plateau_prominence_candidates(
    cache: dict[tuple[str, str], DayRecord],
    source: str = "plateau_prominence",
    min_peak_solar: float = 2.0,
    solar_frac: float = 0.20,
    local_window_steps: int = 4,
    rel_plateau_tol: float = 0.025,
    abs_plateau_tol: float = 0.10,
    min_prominence_mw: float = 0.50,
) -> pd.DataFrame:
    rows = []
    for rec in cache.values():
        if not np.isfinite(rec.solar_peak) or rec.solar_peak < min_peak_solar:
            continue
        search_mask = rec.daytime & (rec.solar >= max(min_peak_solar, solar_frac * rec.solar_peak)) & np.isfinite(rec.net)
        search_idx = np.flatnonzero(search_mask)
        if len(search_idx) < 5:
            continue
        local_peak_mask = np.zeros(len(rec.net), dtype=bool)
        for i in search_idx:
            lo = max(0, int(i) - local_window_steps)
            hi = min(len(rec.net), int(i) + local_window_steps + 1)
            local = rec.net[lo:hi]
            local = local[np.isfinite(local)]
            if len(local) == 0:
                continue
            local_max = float(np.nanmax(local))
            tol = max(abs_plateau_tol, abs(local_max) * rel_plateau_tol)
            if rec.net[i] >= local_max - tol:
                local_peak_mask[i] = True
        cid = 0
        seen = set()
        for region in runs_from_mask(local_peak_mask & search_mask):
            peak_idx = int(region[len(region) // 2])
            index = np.arange(len(rec.net))
            left_pool = np.flatnonzero(rec.daytime & np.isfinite(rec.net) & (index < peak_idx))
            right_pool = np.flatnonzero(rec.daytime & np.isfinite(rec.net) & (index > peak_idx))
            if len(left_pool) == 0 or len(right_pool) == 0:
                continue
            left_idx = int(left_pool[np.nanargmin(rec.net[left_pool])])
            right_idx = int(right_pool[np.nanargmin(rec.net[right_pool])])
            if left_idx >= peak_idx or right_idx <= peak_idx:
                continue
            left_min = float(rec.net[left_idx])
            right_min = float(rec.net[right_idx])
            peak_level = float(np.nanmax(rec.net[region]))
            prominence = peak_level - max(left_min, right_min)
            if prominence < min_prominence_mw:
                continue
            start_idx = min(left_idx + 1, len(rec.ts) - 1)
            end_idx = max(right_idx - 1, 0)
            extra = {
                "peak_time": rec.ts.iloc[peak_idx],
                "plateau_start": rec.ts.iloc[int(region[0])],
                "plateau_end": rec.ts.iloc[int(region[-1])],
                "peak_net_load_MW": peak_level,
                "prominence_MW": prominence,
                "solar_peak_MW": float(rec.solar_peak),
            }
            row = candidate_row(rec, source, cid, rec.ts.iloc[start_idx], rec.ts.iloc[end_idx], extra)
            if row is None:
                continue
            key = (row["pred_start"], row["pred_end"])
            if key in seen:
                continue
            rows.append(row)
            seen.add(key)
            cid += 1
    return pd.DataFrame(rows)

def combine_candidates(variant_id: str, *frames: pd.DataFrame) -> pd.DataFrame:
    valid = [f for f in frames if f is not None and not f.empty]
    if not valid:
        return pd.DataFrame(columns=["substation_id", "date", "candidate_source", "candidate_id", "pred_start", "pred_end"])
    out = pd.concat(valid, ignore_index=True, sort=False)
    out = out.drop_duplicates(["substation_id", "date", "pred_start", "pred_end"]).reset_index(drop=True)
    out["candidate_source"] = variant_id
    out["candidate_id"] = out.groupby(["substation_id", "date"]).cumcount()
    return out

## 6. Focus Scope Selection

        Focus mode first evaluates current m7 on the poor Beta sites, then selects RPF days in the requested failure groups. This gives us a small but high-value lab set for quick experiments.

In [ ]:
beta_focus_m7 = filter_candidates_to_cache(beta_m7, beta_focus_site_cache)
_, _, beta_focus_current_day_all = evaluate_candidates(beta_focus_site_cache, beta_focus_m7, "BetaFocusSites", "current_m7")

beta_focus_current_rpf = beta_focus_current_day_all.loc[beta_focus_current_day_all["label_day"]].copy()
beta_focus_current_rpf["failure_group"] = np.select(
    [
        beta_focus_current_rpf["strict_hit"],
        (~beta_focus_current_rpf["strict_hit"]) & (beta_focus_current_rpf["best_iou"] >= IOU_HIT_THRESHOLD),
        (~beta_focus_current_rpf["strict_hit"]) & beta_focus_current_rpf["candidate_day"] & (beta_focus_current_rpf["best_iou"] < IOU_HIT_THRESHOLD),
        (~beta_focus_current_rpf["candidate_day"]),
    ],
    ["strict_oracle_hit", "boundary_mismatch", "poor_overlap", "no_candidate"],
    default="other",
)

if RUN_MODE == "focus":
    focus_rows = beta_focus_current_rpf.loc[beta_focus_current_rpf["failure_group"].isin(FOCUS_FAILURE_GROUPS)].copy()
    focus_keys = set(zip(focus_rows["substation_id"], focus_rows["date"].astype(str)))
    focus_keys.update(REQUIRED_EXAMPLE_DAYS)
    beta_run_cache = build_day_cache(beta, keys=focus_keys)
    run_caches = {"BetaFocus": beta_run_cache}
    print("Focus site-days:", len(beta_run_cache))
    print("Focus RPF days:", sum(rec.label_day for rec in beta_run_cache.values()))
else:
    alpha_cache = build_day_cache(alpha)
    beta_cache = build_day_cache(beta)
    run_caches = {"Alpha": alpha_cache, "Beta": beta_cache}
    focus_rows = beta_focus_current_rpf.loc[beta_focus_current_rpf["failure_group"].isin(FOCUS_FAILURE_GROUPS)].copy()
    print("Full mode cached site-days:", {name: len(cache) for name, cache in run_caches.items()})

focus_rows.to_csv(CSV_DIR / f"{OUTPUT_PREFIX}00_current_m7_focus_failure_rows.csv", index=False)
display(
    beta_focus_current_rpf.groupby(["substation_id", "failure_group"], as_index=False)
    .size()
    .pivot_table(index="substation_id", columns="failure_group", values="size", fill_value=0)
)

## 7. Build Candidate Variants

        Focus mode builds only the relevant plateau variants. Full mode also includes the older broad diagnostic variants.

In [ ]:
def build_variants_for_cache(cache: dict[tuple[str, str], DayRecord], m7_candidates: pd.DataFrame, include_all: bool) -> dict[str, pd.DataFrame]:
    current = filter_candidates_to_cache(m7_candidates, cache)
    plateau = generate_plateau_prominence_candidates(cache, source="plateau_prominence")
    plateau_loose = generate_plateau_prominence_candidates(
        cache,
        source="plateau_prominence_loose",
        solar_frac=0.15,
        local_window_steps=6,
        rel_plateau_tol=0.035,
        abs_plateau_tol=0.15,
        min_prominence_mw=0.25,
    )
    variants = {
        "current_m7": current,
        "plateau_prominence": plateau,
        "plateau_prominence_loose": plateau_loose,
        "current_plus_plateau": combine_candidates("current_plus_plateau", current, plateau),
        "current_plus_plateau_loose": combine_candidates("current_plus_plateau_loose", current, plateau_loose),
    }
    if include_all:
        solar_peak = generate_solar_peak_windows(cache)
        solar_high = generate_solar_high_segments(cache)
        comove = generate_comovement_segments(cache)
        pseudo = generate_pseudoload_stable_segments(cache)
        variants.update({
            "solar_peak_windows": solar_peak,
            "solar_high_segments": solar_high,
            "comovement_segments": comove,
            "pseudoload_stable_segments": pseudo,
            "current_plus_solar_high": combine_candidates("current_plus_solar_high", current, solar_high),
            "current_plus_comovement": combine_candidates("current_plus_comovement", current, comove),
            "current_plus_pseudoload": combine_candidates("current_plus_pseudoload", current, pseudo),
            "combined_shape_probe": combine_candidates("combined_shape_probe", current, solar_high, comove, pseudo),
            "combined_shape_plus_plateau": combine_candidates("combined_shape_plus_plateau", current, solar_high, comove, pseudo, plateau),
            "combined_shape_plus_plateau_loose": combine_candidates("combined_shape_plus_plateau_loose", current, solar_high, comove, pseudo, plateau_loose),
        })
    else:
        solar_high = generate_solar_high_segments(cache)
        comove = generate_comovement_segments(cache)
        pseudo = generate_pseudoload_stable_segments(cache)
        variants["combined_shape_plus_plateau"] = combine_candidates("combined_shape_plus_plateau", current, solar_high, comove, pseudo, plateau)
    return variants

all_variant_tables = {}
size_rows = []
for dataset, cache in run_caches.items():
    base_m7 = alpha_m7 if dataset == "Alpha" else beta_m7
    variants = build_variants_for_cache(cache, base_m7, include_all=(RUN_MODE == "full"))
    all_variant_tables[dataset] = variants
    for variant_id, table in variants.items():
        size_rows.append({
            "dataset": dataset,
            "variant_id": variant_id,
            "candidates": len(table),
            "candidate_days": table[["substation_id", "date"]].drop_duplicates().shape[0] if not table.empty else 0,
        })

variant_sizes = pd.DataFrame(size_rows)
variant_sizes.to_csv(CSV_DIR / f"{OUTPUT_PREFIX}01_candidate_variant_sizes.csv", index=False)
display(variant_sizes)

## 8. Oracle Metrics

        These are candidate-generator oracle metrics. In focus mode, they apply only to the selected failure-case site-days. In full mode, they are complete Alpha/Beta metrics.

In [ ]:
overall_rows = []
site_rows = []
day_frames = []

for dataset, cache in run_caches.items():
    for variant_id, candidates in all_variant_tables[dataset].items():
        overall, by_site, day = evaluate_candidates(cache, candidates, dataset, variant_id)
        overall_rows.append(overall)
        site_rows.append(by_site)
        day_frames.append(day)

overall_summary = pd.concat(overall_rows, ignore_index=True)
site_summary = pd.concat(site_rows, ignore_index=True)
day_summary = pd.concat(day_frames, ignore_index=True)

overall_summary.to_csv(CSV_DIR / f"{OUTPUT_PREFIX}02_candidate_oracle_summary.csv", index=False)
site_summary.to_csv(CSV_DIR / f"{OUTPUT_PREFIX}03_candidate_oracle_by_site.csv", index=False)
day_summary.to_csv(CSV_DIR / f"{OUTPUT_PREFIX}04_candidate_oracle_by_day.csv", index=False)
if RUN_MODE == "full" and WRITE_FULL_DAY_SUMMARY:
    day_summary.to_csv(CSV_DIR / "03_candidate_oracle_by_day.csv", index=False)
    overall_summary.to_csv(CSV_DIR / "01_candidate_oracle_summary.csv", index=False)
    site_summary.to_csv(CSV_DIR / "02_candidate_oracle_by_site.csv", index=False)

display(
    overall_summary
    .sort_values(["dataset", "strict_recall", "iou50_recall"], ascending=[True, False, False])
    [["dataset", "variant_id", "rpf_days", "strict_recall", "iou50_recall", "mean_best_iou_rpf", "total_candidates", "candidate_days"]]
)

## 9. Recovery Labels And Targeted HTML Examples

        HTML filenames now describe the original current-m7 failure and whether plateau-aware candidates recover the day. This avoids confusion such as `no_candidate` files that now contain recovered plateau candidates.

In [ ]:
def day_lookup(day_summary: pd.DataFrame, dataset: str, site: str, date: str) -> pd.DataFrame:
    return day_summary.loc[
        day_summary["dataset"].eq(dataset)
        & day_summary["substation_id"].eq(site)
        & day_summary["date"].astype(str).eq(str(date))
    ].copy()

def recovery_status(rows: pd.DataFrame) -> tuple[str, str, float, bool]:
    if rows.empty:
        return "not_evaluated", "", 0.0, False
    plateau_ids = ["plateau_prominence", "plateau_prominence_loose", "current_plus_plateau", "current_plus_plateau_loose", "combined_shape_plus_plateau"]
    current = rows.loc[rows["variant_id"].eq("current_m7")]
    if not current.empty and bool(current.iloc[0]["strict_hit"]):
        return "current_m7_hit", "current_m7", float(current.iloc[0]["best_iou"]), True
    strict = rows.loc[rows["variant_id"].isin(plateau_ids) & rows["strict_hit"]]
    if not strict.empty:
        best = strict.sort_values(["best_iou", "candidate_count"], ascending=[False, True]).iloc[0]
        return "recovered_by_plateau", str(best["variant_id"]), float(best["best_iou"]), True
    rough = rows.loc[rows["variant_id"].isin(plateau_ids) & rows["iou50_hit"]]
    if not rough.empty:
        best = rough.sort_values(["best_iou", "candidate_count"], ascending=[False, True]).iloc[0]
        return "rough_overlap_by_plateau", str(best["variant_id"]), float(best["best_iou"]), False
    best = rows.sort_values("best_iou", ascending=False).iloc[0]
    return "not_recovered_by_plateau", str(best["variant_id"]), float(best["best_iou"]), False

def candidates_for_day(candidates: pd.DataFrame, site: str, date: str) -> pd.DataFrame:
    if candidates is None or candidates.empty:
        return pd.DataFrame()
    temp = candidates.copy()
    temp["date"] = temp["date"].astype(str)
    return temp.loc[temp["substation_id"].eq(site) & temp["date"].eq(str(date))].copy()

def save_example_html(site: str, date: str, failure_group: str, status: str, best_variant: str, path: Path) -> Path:
    rec = beta_focus_site_cache.get((site, str(date))) or build_day_cache(beta, keys={(site, str(date))}).get((site, str(date)))
    if rec is None:
        return path
    day = rec.frame
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=day["timestamp"], y=day["net_load_MW"], mode="lines", name="Raw net load", line=dict(color=PALETTE["dark_blue"])), secondary_y=False)
    fig.add_trace(go.Scatter(x=day["timestamp"], y=day["solar_MW"], mode="lines", name="Solar", line=dict(color=PALETTE["orange"])), secondary_y=True)
    fig.add_trace(go.Scatter(x=day["timestamp"], y=day["solar_MW"] - day["net_load_MW"], mode="lines", name="Pseudo-load", line=dict(color=PALETTE["grey"], dash="dot")), secondary_y=False)

    labelled = day.loc[day["label_interval"]]
    if not labelled.empty:
        fig.add_vrect(x0=labelled["timestamp"].iloc[0], x1=labelled["timestamp"].iloc[-1], fillcolor="rgba(235,147,44,0.20)", line_width=0, annotation_text="manual")

    variants = all_variant_tables.get("BetaFocus") or all_variant_tables.get("Beta")
    overlay_order = [
        ("current_m7", "rgba(34,48,61,0.14)", PALETTE["dark_blue"], "current m7"),
        ("plateau_prominence_loose", "rgba(235,147,44,0.12)", PALETTE["orange"], "plateau"),
        ("current_plus_plateau_loose", "rgba(92,125,153,0.10)", PALETTE["light_grey"], "m7+plateau"),
        ("combined_shape_plus_plateau", "rgba(47,77,103,0.08)", PALETTE["grey"], "combined+plateau"),
    ]
    for variant_id, fill, line_color, label in overlay_order:
        if variant_id not in variants:
            continue
        cand = candidates_for_day(variants[variant_id], site, date)
        for idx, row in cand.iterrows():
            fig.add_vrect(
                x0=row["pred_start"],
                x1=row["pred_end"],
                fillcolor=fill,
                line_width=1,
                line_color=line_color,
                annotation_text=label if idx == cand.index[0] else None,
            )
    fig.update_layout(
        title=f"{site} {date} | current m7: {failure_group} | {status} | best: {best_variant}",
        template="plotly_white",
        font=dict(family="Arial", color=PALETTE["dark_blue"]),
        height=560,
    )
    fig.update_yaxes(title_text="Net load / pseudo-load (MW)", secondary_y=False)
    fig.update_yaxes(title_text="Solar (MW)", secondary_y=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path)
    return path

example_source = focus_rows.copy()
if RUN_MODE == "full":
    example_source = beta_focus_current_rpf.loc[beta_focus_current_rpf["failure_group"].isin(FOCUS_FAILURE_GROUPS)].copy()

example_rows = (
    example_source
    .sort_values(["substation_id", "failure_group", "best_iou", "date"], ascending=[True, True, False, True])
    .groupby(["substation_id", "failure_group"], as_index=False)
    .head(EXAMPLES_PER_SITE_GROUP)
)
required_df = pd.DataFrame([{"substation_id": site, "date": date, "failure_group": "required"} for site, date in REQUIRED_EXAMPLE_DAYS])
example_rows = pd.concat([example_rows, required_df], ignore_index=True, sort=False).drop_duplicates(["substation_id", "date"])

html_rows = []
if WRITE_HTML:
    html_dataset = "BetaFocus" if "BetaFocus" in set(day_summary["dataset"]) else "Beta"
    for _, row in example_rows.iterrows():
        site = str(row["substation_id"])
        date = str(row["date"])
        current_rows = beta_focus_current_rpf.loc[beta_focus_current_rpf["substation_id"].eq(site) & beta_focus_current_rpf["date"].astype(str).eq(date)]
        failure_group = str(current_rows["failure_group"].iloc[0]) if not current_rows.empty else str(row.get("failure_group", "unknown"))
        rows = day_lookup(day_summary, html_dataset, site, date)
        status, best_variant, best_iou, strict_hit = recovery_status(rows)
        filename = f"current_m7_{failure_group}__{status}__{site}__{date}.html"
        path = save_example_html(site, date, failure_group, status, best_variant, HTML_DIR / filename)
        html_rows.append({
            "substation_id": site,
            "date": date,
            "current_m7_failure_group": failure_group,
            "plateau_status": status,
            "best_variant": best_variant,
            "best_iou": best_iou,
            "strict_hit": strict_hit,
            "html_file": path.name,
        })

html_index = pd.DataFrame(html_rows)
html_index.to_csv(CSV_DIR / f"{OUTPUT_PREFIX}05_html_example_index.csv", index=False)
display(html_index)

## 10. Figures And Manifest

        Figures are written with `focus_` prefixes in focus mode. The manifest records the mode and the best candidate-generator recall in the current run.

In [ ]:
def save_recall_volume_figure(summary: pd.DataFrame, path: Path) -> Path:
    fig, ax = plt.subplots(figsize=(8.2, 5.2))
    for dataset, group in summary.groupby("dataset"):
        ax.scatter(group["mean_candidates_per_day"], group["strict_recall"], s=70, label=f"{dataset} strict", color=PALETTE["orange"], edgecolor=PALETTE["dark_blue"])
        for _, row in group.iterrows():
            ax.text(row["mean_candidates_per_day"] + 0.02, row["strict_recall"], row["variant_id"], fontsize=8)
    ax.set_xlabel("Mean candidates per site-day")
    ax.set_ylabel("RPF day strict candidate recall")
    ax.set_ylim(0, 1.05)
    ax.set_axisbelow(True)
    ax.grid(color=PALETTE["light_white"], linewidth=0.8)
    ax.legend(frameon=False)
    ax.set_title("Candidate recall versus candidate volume")
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path

def save_site_recall_figure(site_summary: pd.DataFrame, path: Path) -> Path:
    data = site_summary.loc[site_summary["dataset"].str.contains("Beta")].copy()
    keep = ["current_m7", "current_plus_plateau_loose", "combined_shape_plus_plateau"]
    data = data.loc[data["variant_id"].isin(keep)]
    pivot = data.pivot_table(index="substation_id", columns="variant_id", values="strict_recall", aggfunc="first").fillna(0)
    if "current_m7" in pivot:
        pivot = pivot.sort_values("current_m7")
    x = np.arange(len(pivot))
    width = 0.25
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    ax.bar(x - width, pivot.get("current_m7", 0), width, label="Current m7", color=PALETTE["dark_blue"])
    ax.bar(x, pivot.get("current_plus_plateau_loose", 0), width, label="m7 + plateau", color=PALETTE["light_grey"])
    ax.bar(x + width, pivot.get("combined_shape_plus_plateau", 0), width, label="combined + plateau", color=PALETTE["orange"])
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Strict candidate recall")
    ax.set_title("Candidate recall by Beta site")
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=PALETTE["light_white"], linewidth=0.8)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path

fig1 = save_recall_volume_figure(overall_summary, FIGURE_DIR / f"{OUTPUT_PREFIX}fig01_candidate_recall_vs_volume.png")
fig2 = save_site_recall_figure(site_summary, FIGURE_DIR / f"{OUTPUT_PREFIX}fig02_site_candidate_recall.png")

best = overall_summary.sort_values(["strict_recall", "iou50_recall"], ascending=False).iloc[0]
manifest = {
    "mode": "candidate_generator_oracle_lab",
    "run_mode": RUN_MODE,
    "publication_ready": False,
    "elapsed_seconds": time.perf_counter() - t0,
    "focus_sites": FOCUS_SITES,
    "focus_failure_groups": FOCUS_FAILURE_GROUPS,
    "examples_per_site_group": EXAMPLES_PER_SITE_GROUP,
    "write_html": WRITE_HTML,
    "write_full_day_summary": WRITE_FULL_DAY_SUMMARY,
    "best_dataset": str(best["dataset"]),
    "best_variant": str(best["variant_id"]),
    "best_strict_recall": float(best["strict_recall"]),
    "best_iou50_recall": float(best["iou50_recall"]),
    "outputs": {
        "csv": sorted(p.name for p in CSV_DIR.glob(f"{OUTPUT_PREFIX}*.csv")),
        "figures": [fig1.name, fig2.name],
        "html": sorted(p.name for p in HTML_DIR.glob("*.html")) if WRITE_HTML else [],
    },
}
manifest_path = MANIFEST_DIR / f"{OUTPUT_PREFIX}run_manifest.json"
with manifest_path.open("w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2, default=str)
display(pd.DataFrame([manifest]))